In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


statistics_player = r"C:\Users\Lucas\OneDrive\Desktop\ia\safa e transfermarkt\statistics_player.csv"
df = pd.read_csv(statistics_player)

In [23]:
df

,id_partida,data_jogo,campeonato,team,team_name,name,position,jerseyNumber,height,country,...,shotOffTarget,interceptionWon,keyPass,dispossessed,blockedScoringAttempt,bigChanceCreated,penaltyWon,totalOffside,goalsPrevented,penaltyConceded
0,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Rafael,G,23.0,192.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.1379,NaN
1,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Rafinha,D,13.0,172.0,Brazil,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Robert Arboleda,D,5.0,187.0,Ecuador,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Alan Franco,D,28.0,183.0,Argentina,...,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Welington,D,6.0,170.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39856,12146489,2024-10-12,Brasileirão Série B,away,Operário-PR,Joseph,D,30.0,186.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39857,12146489,2024-10-12,Brasileirão Série B,away,Operário-PR,Savio,D,23.0,168.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39858,12146489,2024-10-12,Brasileirão Série B,away,Operário-PR,Lucas Hipólito,D,13.0,172.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39859,12146489,2024-10-12,Brasileirão Série B,away,Operário-PR,Índio,M,5.0,185.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df = df.copy()

df["goals"] = df["goals"].fillna(0)
df["goalAssist"] = df["goalAssist"].fillna(0)
df["expectedGoals"] = df["expectedGoals"].fillna(0)
df["expectedAssists"] = df["expectedAssists"].fillna(0)
df["rating"] = df["rating"].fillna(0)
df["minutesPlayed"] = df["minutesPlayed"].fillna(0)

df["pass_accuracy"] = df["accuratePass"] / (df["totalPass"] + 1)
df["long_ball_accuracy"] = df["accurateLongBalls"] / (df["totalLongBalls"] + 1)
df["duel_win_rate"] = df["duelWon"] / (df["duelWon"] + df["duelLost"] + 1)
df["aerial_win_rate"] = df["aerialWon"] / (df["aerialWon"] + df["aerialLost"] + 1)

df["goal_participation"] = df["goals"] + df["goalAssist"]
df["expected_participation"] = df["expectedGoals"] + df["expectedAssists"]

df["goals_per_90"] = df["goals"] / (df["minutesPlayed"] / 90 + 1)
df["assists_per_90"] = df["goalAssist"] / (df["minutesPlayed"] / 90 + 1)
df["xg_per_90"] = df["expectedGoals"] / (df["minutesPlayed"] / 90 + 1)
df["xa_per_90"] = df["expectedAssists"] / (df["minutesPlayed"] / 90 + 1)

In [25]:
df["score_potencial"] = (
    df["rating"] * 0.45 +
    df["expectedGoals"] * 1.5 +
    df["expectedAssists"] * 1.5 +
    df["goals"] * 1.2 +
    df["goalAssist"] * 1.0 +
    df["duel_win_rate"] * 0.5 +
    df["pass_accuracy"] * 0.5
)

df["craque"] = (df["score_potencial"] >= df["score_potencial"].quantile(0.80)).astype(int)


In [26]:
colunas_remover = [
    "craque",
    "score_potencial",
    "name",
    "team_name",
    "dateOfBirth",
    "id_partida",
    "data_jogo"
]

X = df.drop(columns=[c for c in colunas_remover if c in df.columns])
y = df["craque"]




In [27]:
categoricas = ["campeonato", "team", "position", "country"]
categoricas = [c for c in categoricas if c in X.columns]

numericas = [c for c in X.columns if c not in categoricas]


preprocessador = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)
    ]
)


In [28]:
modelo = Pipeline(steps=[
    ("preprocessador", preprocessador),
    ("classificador", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])



In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

modelo.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessador', ...), ('classificador', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [30]:
y_pred = modelo.predict(X_test)

print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7147
           1       0.95      0.91      0.93       826

    accuracy                           0.99      7973
   macro avg       0.97      0.95      0.96      7973
weighted avg       0.99      0.99      0.99      7973



In [31]:
df["probabilidade_craque"] = modelo.predict_proba(X)[:, 1]

ranking = df[[
    "name",
    "team_name",
    "position",
    "age",
    "rating",
    "goals",
    "goalAssist",
    "expectedGoals",
    "expectedAssists",
    "probabilidade_craque"
]].sort_values(by="probabilidade_craque", ascending=False)

print(ranking.head(20))

                     name            team_name position   age  rating  goals  \
19017       Eduardo Sasha  Red Bull Bragantino        F  32.0     7.4    0.0   
17649       Luiz Fernando  Atlético Goianiense        M  27.0     7.2    1.0   
6736               Wesley        Internacional        M  25.0     8.0    0.0   
31775           Alex Arce                  LDU        F  29.0     7.6    1.0   
7710     Gustavo Coutinho         Sport Recife        F  25.0     7.5    1.0   
37964           Guilherme               Santos        M  29.0     8.4    0.0   
34598         Rafael Gava                Goiás        M  31.0     7.8    1.0   
1202                 Hulk     Atlético Mineiro        F  38.0     7.4    0.0   
30430              Marlon              Guarani        F  27.0     7.2    1.0   
38282              Wesley        Internacional        M  25.0     7.8    2.0   
23019         Lucas Moura            São Paulo        M  32.0     7.6    1.0   
38143         Lucas Moura            São

In [32]:
df['probabilidade_craque']

0        0.000000
1        0.003333
2        0.963333
3        0.146667
4        0.000000
           ...   
39856    0.000000
39857    0.000000
39858    0.000000
39859    0.000000
39860    0.000000
Name: probabilidade_craque, Length: 39861, dtype: float64

UTILIZANDO DA PRÓPRIA IA PARA DECIDIR SE É OU NÃO É

In [33]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans

In [34]:
caminho = r"C:\Users\Lucas\OneDrive\Desktop\ia\safa e transfermarkt\statistics_player.csv"

df = pd.read_csv(caminho)
df = df.copy()

df.head()

,id_partida,data_jogo,campeonato,team,team_name,name,position,jerseyNumber,height,country,...,shotOffTarget,interceptionWon,keyPass,dispossessed,blockedScoringAttempt,bigChanceCreated,penaltyWon,totalOffside,goalsPrevented,penaltyConceded
0,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Rafael,G,23.0,192.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.1379,NaN
1,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Rafinha,D,13.0,172.0,Brazil,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Robert Arboleda,D,5.0,187.0,Ecuador,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Alan Franco,D,28.0,183.0,Argentina,...,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12117250,2024-09-29,Brasileirão Série A,home,São Paulo,Welington,D,6.0,170.0,Brazil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
colunas_zero = [
    "goals", "goalAssist", "expectedGoals", "expectedAssists",
    "rating", "minutesPlayed", "accuratePass", "totalPass",
    "accurateLongBalls", "totalLongBalls", "duelWon", "duelLost",
    "aerialWon", "aerialLost"
]

for col in colunas_zero:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [36]:
colunas_numericas = df.select_dtypes(include=np.number).columns.tolist()

# remover colunas que não devem ser somadas
nao_agrupar = ["id_partida"]
colunas_numericas = [c for c in colunas_numericas if c not in nao_agrupar]

df_agregado = df.groupby("name").agg({
    # SOMA (volume total)
    "goals": "sum",
    "goalAssist": "sum",
    "expectedGoals": "sum",
    "expectedAssists": "sum",
    "minutesPlayed": "sum",
    "duelWon": "sum",
    "duelLost": "sum",
    "aerialWon": "sum",
    "aerialLost": "sum",
    "accuratePass": "sum",
    "totalPass": "sum",
    "accurateLongBalls": "sum",
    "totalLongBalls": "sum",

    # MÉDIA (qualidade)
    "rating": "mean",

    # PEGAR PRIMEIRO (info fixa)
    "team_name": "first",
    "position": "first",
    "age": "first",
    "country": "first",
    "team": "first",
    "campeonato": "first"

}).reset_index()

df = df_agregado.copy()

print("Jogadores únicos:", len(df))
df.head()

Jogadores únicos: 3027


,name,goals,goalAssist,expectedGoals,expectedAssists,minutesPlayed,duelWon,duelLost,aerialWon,aerialLost,...,totalPass,accurateLongBalls,totalLongBalls,rating,team_name,position,age,country,team,campeonato
0,Aamet Calderón,0.0,0.0,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,Universitario,G,26.0,Peru,home,"Copa Libertadores, Group D"
1,Aaron Anselmino,1.0,0.0,0.0000,0.000000,170.0,6.0,2.0,3.0,0.0,...,97.0,5.0,11.0,7.350000,Boca Juniors,D,19.0,Argentina,away,"Copa Sudamericana, Group D"
2,Aaron Molinas,0.0,0.0,0.0000,0.000000,136.0,8.0,6.0,0.0,2.0,...,61.0,6.0,7.0,5.325000,Defensa y Justicia,M,24.0,Argentina,away,"Copa Sudamericana, Group A"
3,Abdiel Ayarza,0.0,0.0,0.3913,0.026524,156.0,6.0,10.0,2.0,1.0,...,45.0,3.0,3.0,6.550000,The Strongest,M,32.0,Panama,away,"Copa Libertadores, Knockout stage"
4,Abel Luciatti,0.0,1.0,0.0000,0.000000,687.0,34.0,20.0,23.0,9.0,...,254.0,14.0,27.0,6.311111,Lanús,D,31.0,Argentina,away,"Copa Sudamericana, Group G"


In [37]:
df["pass_accuracy"] = df["accuratePass"] / (df["totalPass"] + 1)
df["long_ball_accuracy"] = df["accurateLongBalls"] / (df["totalLongBalls"] + 1)

df["duel_win_rate"] = df["duelWon"] / (df["duelWon"] + df["duelLost"] + 1)
df["aerial_win_rate"] = df["aerialWon"] / (df["aerialWon"] + df["aerialLost"] + 1)

df["goal_participation"] = df["goals"] + df["goalAssist"]
df["expected_participation"] = df["expectedGoals"] + df["expectedAssists"]

df["goals_per_90"] = df["goals"] / (df["minutesPlayed"] / 90 + 1)
df["assists_per_90"] = df["goalAssist"] / (df["minutesPlayed"] / 90 + 1)
df["xg_per_90"] = df["expectedGoals"] / (df["minutesPlayed"] / 90 + 1)
df["xa_per_90"] = df["expectedAssists"] / (df["minutesPlayed"] / 90 + 1)

df.head()

,name,goals,goalAssist,expectedGoals,expectedAssists,minutesPlayed,duelWon,duelLost,aerialWon,aerialLost,...,pass_accuracy,long_ball_accuracy,duel_win_rate,aerial_win_rate,goal_participation,expected_participation,goals_per_90,assists_per_90,xg_per_90,xa_per_90
0,Aamet Calderón,0.0,0.0,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.00000,0.000000,0.000000
1,Aaron Anselmino,1.0,0.0,0.0000,0.000000,170.0,6.0,2.0,3.0,0.0,...,0.867347,0.416667,0.666667,0.75000,1.0,0.000000,0.346154,0.00000,0.000000,0.000000
2,Aaron Molinas,0.0,0.0,0.0000,0.000000,136.0,8.0,6.0,0.0,2.0,...,0.838710,0.750000,0.533333,0.00000,0.0,0.000000,0.000000,0.00000,0.000000,0.000000
3,Abdiel Ayarza,0.0,0.0,0.3913,0.026524,156.0,6.0,10.0,2.0,1.0,...,0.826087,0.750000,0.352941,0.50000,0.0,0.417824,0.000000,0.00000,0.143159,0.009704
4,Abel Luciatti,0.0,1.0,0.0000,0.000000,687.0,34.0,20.0,23.0,9.0,...,0.847059,0.500000,0.618182,0.69697,1.0,0.000000,0.000000,0.11583,0.000000,0.000000


In [38]:
colunas_remover = [
    "name",
    "team_name",
    "dateOfBirth",
    "id_partida",
    "data_jogo"
]

X = df.drop(columns=[c for c in colunas_remover if c in df.columns])

categoricas = ["campeonato", "team", "position", "country"]
categoricas = [c for c in categoricas if c in X.columns]

numericas = [c for c in X.columns if c not in categoricas]

print("Numéricas:", len(numericas))
print("Categóricas:", categoricas)

Numéricas: 25
Categóricas: ['campeonato', 'team', 'position', 'country']


In [39]:
preprocessador = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numericas),

        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)
    ]
)

In [40]:
modelo_cluster = Pipeline(steps=[
    ("preprocessador", preprocessador),
    ("cluster", KMeans(
        n_clusters=4,
        random_state=42,
        n_init=10
    ))
])

df["grupo_ia"] = modelo_cluster.fit_predict(X)

df[["name", "grupo_ia"]].head()

,name,grupo_ia
0,Aamet Calderón,0
1,Aaron Anselmino,3
2,Aaron Molinas,3
3,Abdiel Ayarza,3
4,Abel Luciatti,3


In [41]:
analise_grupos = df.groupby("grupo_ia")[[
    "rating",
    "goals_per_90",
    "assists_per_90",
    "xg_per_90",
    "xa_per_90",
    "pass_accuracy",
    "duel_win_rate",
    "aerial_win_rate",
    "minutesPlayed"
]].mean()

analise_grupos

,rating,goals_per_90,assists_per_90,xg_per_90,xa_per_90,pass_accuracy,duel_win_rate,aerial_win_rate,minutesPlayed
grupo_ia,,,,,,,,,
0,0.873819,0.002302,0.000586,0.001907,0.001256,0.197415,0.068508,0.015990,19.173565
1,6.302693,0.256917,0.146703,0.183267,0.092464,0.779346,0.458187,0.382679,1834.786517
2,6.017657,0.057575,0.043333,0.015546,0.015467,0.806437,0.558435,0.553051,2119.735795
3,4.929123,0.066359,0.050151,0.028846,0.019018,0.747057,0.475302,0.386756,380.385011


In [42]:
analise_grupos["indice_potencial_grupo"] = analise_grupos[[
    "rating",
    "goals_per_90",
    "assists_per_90",
    "xg_per_90",
    "xa_per_90",
    "pass_accuracy",
    "duel_win_rate"
]].mean(axis=1)

melhor_grupo = analise_grupos["indice_potencial_grupo"].idxmax()

print("Melhor grupo:", melhor_grupo)
analise_grupos

Melhor grupo: 1


,rating,goals_per_90,assists_per_90,xg_per_90,xa_per_90,pass_accuracy,duel_win_rate,aerial_win_rate,minutesPlayed,indice_potencial_grupo
grupo_ia,,,,,,,,,,
0,0.873819,0.002302,0.000586,0.001907,0.001256,0.197415,0.068508,0.015990,19.173565,0.163685
1,6.302693,0.256917,0.146703,0.183267,0.092464,0.779346,0.458187,0.382679,1834.786517,1.174225
2,6.017657,0.057575,0.043333,0.015546,0.015467,0.806437,0.558435,0.553051,2119.735795,1.073493
3,4.929123,0.066359,0.050151,0.028846,0.019018,0.747057,0.475302,0.386756,380.385011,0.902265


In [43]:
df["possivel_craque"] = (df["grupo_ia"] == melhor_grupo).astype(int)

df[["name", "grupo_ia", "possivel_craque"]].head()

,name,grupo_ia,possivel_craque
0,Aamet Calderón,0,0
1,Aaron Anselmino,3,0
2,Aaron Molinas,3,0
3,Abdiel Ayarza,3,0
4,Abel Luciatti,3,0


In [44]:
df["indice_potencial"] = df[[
    "rating",
    "goals_per_90",
    "assists_per_90",
    "xg_per_90",
    "xa_per_90",
    "pass_accuracy",
    "duel_win_rate"
]].rank(pct=True).mean(axis=1)

ranking = df[[
    "name",
    "team_name",
    "position",
    "age",
    "minutesPlayed",
    "rating",
    "goals",
    "goalAssist",
    "expectedGoals",
    "expectedAssists",
    "goals_per_90",
    "assists_per_90",
    "xg_per_90",
    "xa_per_90",
    "grupo_ia",
    "possivel_craque",
    "indice_potencial"
]].sort_values(by="indice_potencial", ascending=False)

ranking.head(30)

,name,team_name,position,age,minutesPlayed,rating,goals,goalAssist,expectedGoals,expectedAssists,goals_per_90,assists_per_90,xg_per_90,xa_per_90,grupo_ia,possivel_craque,indice_potencial
2469,Raphael Veiga,Palmeiras,M,29.0,2556.0,7.127027,7.0,6.0,7.6912,5.494607,0.238095,0.204082,0.261605,0.186891,1,1,0.904007
2094,Matheus Pereira,Cruzeiro,F,28.0,2836.0,7.550000,8.0,7.0,3.8918,4.062387,0.246070,0.215311,0.119707,0.124954,1,1,0.903393
1397,Jhon Arias,Fluminense,M,27.0,2900.0,7.309091,7.0,6.0,4.7815,3.694278,0.210702,0.180602,0.143925,0.111199,1,1,0.897659
1032,Ganso,Fluminense,M,35.0,2777.0,6.984615,5.0,8.0,1.9745,3.393920,0.156958,0.251134,0.061983,0.106541,1,1,0.895158
1163,Gustavo Scarpa,Atlético Mineiro,M,30.0,3128.0,7.072727,8.0,11.0,4.6624,7.173680,0.223741,0.307644,0.130397,0.200631,1,1,0.894828
1060,Gerson,Flamengo,M,27.0,3339.0,7.283333,4.0,5.0,2.9804,4.688622,0.104987,0.131234,0.078226,0.123061,1,1,0.888173
915,Ferreira,São Paulo,F,26.0,1333.0,6.393548,5.0,2.0,2.9001,2.268677,0.316233,0.126493,0.183422,0.143486,1,1,0.885011
1103,Guido Carrillo,Estudiantes de La Plata,F,33.0,125.0,7.125000,1.0,1.0,1.1677,0.229795,0.418605,0.418605,0.488805,0.096193,3,0,0.882486
2464,Ramón Sosa,Talleres,M,25.0,287.0,6.040000,1.0,1.0,0.8271,0.829083,0.238727,0.238727,0.197451,0.197924,3,0,0.881778
1899,Luiz Araújo,Flamengo,M,28.0,2200.0,7.127027,5.0,7.0,3.3231,5.605443,0.196507,0.275109,0.130602,0.220301,1,1,0.880622


In [ ]:
df_final = ranking.iloc[ranking['possivel_craque'] == 1]
df_final